# Bài thực hành Deep Learning: Autoencoder

Notebook này minh họa cách dùng Autoencoder để nén ảnh, tái tạo ảnh và dùng Encoder kết hợp Dense classifier để nhận dạng nhãn ảnh.

## 1. Mục tiêu bài thực hành

- Cài đặt Autoencoder cho CIFAR10, Cat/Dog, Fashion-MNIST và Nam/Nữ.
- Hiển thị ảnh gốc và ảnh tái tạo.
- Tính reconstruction loss.
- Lấy Encoder làm bộ trích xuất đặc trưng.
- Gắn Dense classifier phía sau Encoder để nhận dạng nhãn ảnh.
- Lưu Autoencoder, Encoder và Classifier riêng.

## 2. Giới thiệu Autoencoder

Autoencoder là mô hình học cách sao chép đầu vào sang đầu ra thông qua một tầng biểu diễn nhỏ hơn. Vì phải nén thông tin, mô hình học được đặc trưng quan trọng của ảnh.

## 3. Encoder, Decoder và Latent Representation

- **Encoder**: nén ảnh thành đặc trưng.
- **Latent Representation**: biểu diễn nén ở giữa mô hình.
- **Decoder**: tái tạo ảnh từ biểu diễn nén.
- **Classifier**: nhận đặc trưng từ Encoder và dự đoán nhãn.

In [ ]:
# 4. Import thư viện
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tensorflow.keras.datasets import cifar10, fashion_mnist
from tensorflow.keras.models import load_model

from utils.labels import CIFAR10_LABELS, FASHION_MNIST_LABELS, CATDOG_LABELS, GENDER_LABELS
from utils.preprocess import (
    MODELS_DIR,
    PLOTS_DIR,
    ensure_project_dirs,
    normalize_images,
    plot_reconstructions,
    predict_image_autoencoder,
    reconstruction_loss,
)

ensure_project_dirs()
EPOCHS = 5
EPOCHS_LIST = [50, 100, 200]
print('Đã import thư viện và tạo thư mục cần thiết.')

## 5. Hàm dùng chung

Các hàm dùng chung nằm trong `utils/preprocess.py` và `utils/model_builders.py`. Notebook dùng lại các hàm này để tránh lặp code.

In [ ]:
def show_images(images, labels=None, class_names=None, n=10, grayscale=False):
    plt.figure(figsize=(n * 1.4, 1.8))
    for i in range(min(n, len(images))):
        plt.subplot(1, n, i + 1)
        if grayscale:
            plt.imshow(images[i].squeeze(), cmap='gray')
        else:
            plt.imshow(images[i])
        if labels is not None and class_names is not None:
            plt.title(class_names[int(labels[i])], fontsize=8)
        plt.axis('off')
    plt.tight_layout()
    plt.show()

def show_reconstruction_if_model_exists(autoencoder_path, images, grayscale=False):
    autoencoder_path = Path(autoencoder_path)
    if not autoencoder_path.exists():
        print('Chưa có model. Vui lòng chạy file train tương ứng trước.')
        return
    autoencoder = load_model(autoencoder_path, compile=False)
    reconstructed = autoencoder.predict(images[:10], verbose=0)
    print('Reconstruction loss:', reconstruction_loss(images[:10], reconstructed))
    plot_reconstructions(images[:10], reconstructed, PLOTS_DIR / 'notebook_reconstruction.png', grayscale=grayscale)
    show_images(images[:10], n=10, grayscale=grayscale)
    show_images(reconstructed[:10], n=10, grayscale=grayscale)

## 6. Bài 1: Autoencoder CIFAR10

CIFAR10 gồm 10 lớp: airplane, automobile, bird, cat, deer, dog, frog, horse, ship, truck. Ảnh đầu vào có kích thước 32x32x3.

In [ ]:
(x_cifar_train, y_cifar_train), (x_cifar_test, y_cifar_test) = cifar10.load_data()
x_cifar_test_norm = normalize_images(x_cifar_test)
show_images(x_cifar_train[:10], y_cifar_train[:10].reshape(-1), CIFAR10_LABELS, n=10)

# Train model CIFAR10. Bỏ dấu # ở dòng dưới nếu muốn train ngay trong notebook.
# %run training/train_cifar10_autoencoder.py

show_reconstruction_if_model_exists(MODELS_DIR / 'ae_cifar10_autoencoder.h5', x_cifar_test_norm, grayscale=False)

## 7. Bài 2: Autoencoder Cat/Dog

Bài này ưu tiên dùng `tensorflow_datasets` với dataset `cats_vs_dogs`. Nếu không tải được, hãy chuẩn bị thư mục `datasets/catdog/train/cat`, `datasets/catdog/train/dog`, `datasets/catdog/val/cat`, `datasets/catdog/val/dog`.

In [ ]:
# Train model Cat/Dog. Dataset có thể tải lâu ở lần đầu.
# %run training/train_catdog_autoencoder.py

print('Nếu chưa train, hãy chạy: python training/train_catdog_autoencoder.py')

## 8. Bài 3: Autoencoder Fashion-MNIST

Fashion-MNIST gồm 10 lớp thời trang. Ảnh đầu vào là grayscale 28x28x1.

In [ ]:
(x_fashion_train, y_fashion_train), (x_fashion_test, y_fashion_test) = fashion_mnist.load_data()
x_fashion_test_norm = normalize_images(x_fashion_test, grayscale=True)
show_images(x_fashion_train[:10], y_fashion_train[:10], FASHION_MNIST_LABELS, n=10, grayscale=True)

# Train model Fashion-MNIST. Bỏ dấu # ở dòng dưới nếu muốn train ngay.
# %run training/train_fashion_autoencoder.py

show_reconstruction_if_model_exists(MODELS_DIR / 'ae_fashion_autoencoder.h5', x_fashion_test_norm, grayscale=True)

## 9. Bài 4: Autoencoder Nam/Nữ

Vui lòng tải dataset khuôn mặt Nam/Nữ public như FairFace, sau đó tách ảnh vào thư mục `datasets/gender/train/male`, `datasets/gender/train/female`, `datasets/gender/val/male`, `datasets/gender/val/female`.

In [ ]:
# Train model Nam/Nữ sau khi đã chuẩn bị dataset.
# %run training/train_gender_autoencoder.py

print('Nếu chưa có dataset, notebook không dừng lỗi. Hãy chuẩn bị dữ liệu theo README.md.')

## 10. So sánh epochs 50, 100, 200

Trong mỗi file train đều có `EPOCHS_LIST = [50, 100, 200]`. Để chạy so sánh, đổi `RUN_EPOCH_COMPARISON = True` trong file train tương ứng.

In [ ]:
comparison_note = pd.DataFrame([
    {'epochs': 50, 'nhan_xet': 'Ảnh tái tạo bắt đầu rõ hơn, thời gian train tăng.'},
    {'epochs': 100, 'nhan_xet': 'Validation loss thường giảm thêm nếu model chưa overfit.'},
    {'epochs': 200, 'nhan_xet': 'Ảnh có thể rõ hơn nhưng tốn thời gian, cần theo dõi validation loss.'},
])
comparison_note

## 11. Lưu mô hình

Mỗi bài lưu 3 model: Autoencoder, Encoder và Classifier. Các file được lưu trong thư mục `models/`.

In [ ]:
summary = pd.DataFrame([
    ['CIFAR10', 'TensorFlow/Keras CIFAR10', '32x32x3', '4x4x128 feature map', 'mse', 'sparse_categorical_crossentropy', 'ae_cifar10_*.h5'],
    ['Cat/Dog', 'TensorFlow Datasets cats_vs_dogs hoặc thư mục local', '64x64x3', '8x8x128 feature map', 'mse', 'binary_crossentropy', 'ae_catdog_*.h5'],
    ['Fashion-MNIST', 'TensorFlow/Keras Fashion-MNIST', '28x28x1', '32', 'mse', 'sparse_categorical_crossentropy', 'ae_fashion_*.h5'],
    ['Nam/Nữ', 'FairFace hoặc thư mục local male/female', '64x64x3', '8x8x128 feature map', 'mse', 'binary_crossentropy', 'ae_gender_*.h5'],
], columns=['Tên bài', 'Dataset', 'Kích thước ảnh', 'Latent dimension', 'Loss Autoencoder', 'Loss Classifier', 'File model đã lưu'])
summary

## 12. Dự đoán ảnh mới

Hàm `predict_image_autoencoder(classifier_path, image_path, target_size, class_names, grayscale=False)` nằm trong `utils/preprocess.py`.

In [ ]:
# Ví dụ dự đoán ảnh mới sau khi đã train model CIFAR10:
# predict_image_autoencoder(
#     classifier_path=MODELS_DIR / 'ae_cifar10_classifier.h5',
#     image_path='static/uploads/example.jpg',
#     target_size=(32, 32),
#     class_names=CIFAR10_LABELS,
#     grayscale=False,
# )

## 13. Tổng kết và kết luận

- Autoencoder học cách nén và tái tạo ảnh.
- Epochs tăng thường giúp ảnh tái tạo rõ hơn nhưng thời gian train lâu hơn.
- CIFAR10 khó hơn Fashion-MNIST vì ảnh màu và nhiều đối tượng hơn.
- Cat/Dog và Nam/Nữ phụ thuộc nhiều vào chất lượng dataset.
- Autoencoder không phải mô hình phân loại trực tiếp, cần kết hợp Encoder với classifier để nhận dạng nhãn.